# Quill — Notebook 05: Controlled Operations Between Knots

In this notebook, one qubit in one Knot controls a gate on a qubit in another Knot. We will first see a deterministic CNOT, then put the control qubit into superposition and observe the correlations.

**Learning goals**
- Explain control and target qubits.
- Write the CNOT transformation with LaTeX.
- Apply a cross-Knot CNOT to `Cord("HI")`.
- Decode measurement outcomes and connect them to the underlying bits.

> A gate acts on qubits, not on the meaning of letters. A character changes only because its encoded bits change.

## 1. Recall the Cord encoding

Each character is encoded as an 8-qubit ASCII register. For `Cord("HI")`, Knot H occupies qubits 0–7 and Knot I occupies qubits 8–15. **Qubit 0 is the least-significant bit (LSB)** of the first character; qubit 8 is the LSB of the second.

| Character | ASCII decimal | Binary, MSB first |
|---|---:|---|
| H | 72 | `01001000` |
| I | 73 | `01001001` |

So H and I differ only in their least-significant bit. That makes them a useful teaching example.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

simulator = AerSimulator()

def encode_cord(text):
    """Prepare ASCII characters in separate 8-qubit registers."""
    qc = QuantumCircuit(8 * len(text), 8 * len(text))
    for char_index, char in enumerate(text):
        value = ord(char)
        for bit in range(8):
            if (value >> bit) & 1:
                qc.x(8 * char_index + bit)
    return qc

def decode_qiskit_bitstring(bitstring, number_of_chars):
    """Decode Qiskit's displayed c[n-1]...c[0] string into ASCII characters.""
    bits_lsb_first = bitstring[::-1]
    chars = []
    for char_index in range(number_of_chars):
        value = sum(int(bits_lsb_first[8 * char_index + bit]) << bit for bit in range(8))
        chars.append(chr(value))
    return ''.join(chars)

base = encode_cord("HI")
print("Qubits:", base.num_qubits)
print("Cord: H then I")
base.draw("mpl")

## 2. A controlled-NOT (CNOT)

CNOT has a **control** qubit and a **target** qubit. It flips the target only when the control is `1`:

$$|c,t\rangle \longmapsto |c, t \oplus c\rangle$$

Here, $\oplus$ means XOR. The control is unchanged.

For example:

$$|0,0\rangle \to |0,0\rangle, \qquad |1,0\rangle \to |1,1\rangle$$

For this notebook, control qubit 0 belongs to Knot H and target qubit 8 belongs to Knot I.

## 3. Deterministic case: control is `0`

In `HI`, the LSB of H is `0`, and the LSB of I is `1`. A CNOT controlled by qubit 0 therefore does not flip qubit 8. The Cord stays `HI`.

In [ ]:
qc = encode_cord("HI")
qc.cx(0, 8)  # control: Knot H's qubit 0; target: Knot I's qubit 0
qc.measure(range(16), range(16))
result = simulator.run(qc, shots=100).result()
counts = result.get_counts()
print(counts)
for bits, count in counts.items():
    print(decode_qiskit_bitstring(bits, 2), count)


## 4. Put the control into superposition

First apply a Hadamard to qubit 0. Since its starting state is $|0\rangle$:

$$H|0\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}$$

The two possible control values now coexist in the state. CNOT correlates the target bit with that control:

$$|HI\rangle \xrightarrow{H_0} \frac{|HI\rangle+|II\rangle}{\sqrt{2}}$$

Then CNOT flips qubit 8 only in the branch where qubit 0 is `1`. In that branch, H's LSB changes H to I, and I's LSB changes I to H:

$$\frac{|HI\rangle+|II\rangle}{\sqrt{2}} \xrightarrow{\mathrm{CNOT}_{0\to8}} \frac{|HI\rangle+|IH\rangle}{\sqrt{2}}$$

This is an entangled state of the two selected qubits. Measuring in the computational basis gives `HI` or `IH`, ideally with equal probability.

In [ ]:
qc = encode_cord("HI")
qc.h(0)       # create a superposition in Knot H's least-significant bit
qc.cx(0, 8)   # that qubit controls Knot I's least-significant bit
qc.measure(range(16), range(16))

result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()
print(counts)

# Aggregate counts by decoded two-character Cord.
cord_counts = {}
for bitstring, count in counts.items():
    cord = decode_qiskit_bitstring(bitstring, 2)
    cord_counts[cord] = cord_counts.get(cord, 0) + count
print(cord_counts)
plot_histogram(cord_counts)

## 5. What to notice

- The CNOT does **not** make every character change. It conditionally flips one target qubit.
- With the control fixed at `0`, the target stays as it was.
- With the control in superposition, the target becomes correlated with it.
- One measurement returns one result; repeated shots reveal the distribution.
- The result is about encoded bits. The gate does not understand the letters H or I.

## 6. Try it yourself

1. Remove `qc.h(0)`. What happens to the outcome distribution?
2. Add `qc.x(0)` before the CNOT but do not add H. Which Cord do you get?
3. Change the target from qubit `8` to qubit `9`. Does the result still decode to just H and I? Why or why not?
4. Try `Cord("AB")` and identify a pair of bit positions to control. Predict the output before running the circuit.

**Next notebook:** Quantum interference — how amplitudes can reinforce or cancel, changing measurement probabilities.